# 11 LoRA 微调 SFT

目标：用一个最小的监督微调任务理解 LoRA adapter、可训练参数比例、训练、保存和加载。


## 1. 安装依赖


In [1]:
from pathlib import Path

base = Path.cwd()
requirements_path = base / "requirements.txt"
advanced_requirements_path = base / "advanced" / "requirements-advanced.txt"

if not requirements_path.exists():
    requirements_path = Path("../requirements.txt")

if not advanced_requirements_path.exists():
    if Path("requirements-advanced.txt").exists():
        advanced_requirements_path = Path("requirements-advanced.txt")
    else:
        advanced_requirements_path = Path("../advanced/requirements-advanced.txt")

print("requirements:", requirements_path)
print("advanced requirements:", advanced_requirements_path)
%pip install -r {requirements_path} -r {advanced_requirements_path}
%pip install --upgrade --force-reinstall --no-cache-dir --no-deps "peft>=0.18.1,<0.19.0"


requirements: ../requirements.txt
advanced requirements: ../advanced/requirements-advanced.txt
ERROR: Could not open requirements file: [Errno 2] 没有那个文件或目录: '../requirements.txt'

[notice] A new release of pip is available: 23.3.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 2. 模型与工具函数


In [ ]:
import gc
import os
from pathlib import Path
from time import perf_counter

import torch
from modelscope import snapshot_download
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

MODEL_ID = os.getenv("MODEL_ID", "Qwen/Qwen2.5-0.5B-Instruct")
MODEL_SOURCE = os.getenv("MODEL_SOURCE", "modelscope").lower()


def resolve_model_path(model_id):
    if Path(model_id).exists():
        return model_id
    if MODEL_SOURCE == "modelscope":
        return snapshot_download(model_id)
    return model_id


def cleanup(*objects):
    for obj in objects:
        del obj
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def cuda_memory(label=""):
    if not torch.cuda.is_available():
        print(label, "cuda unavailable")
        return
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    peak = torch.cuda.max_memory_allocated() / 1024**3
    print(f"{label} allocated={allocated:.2f}GB reserved={reserved:.2f}GB peak={peak:.2f}GB")


MODEL_PATH = resolve_model_path(MODEL_ID)
print("MODEL_ID =", MODEL_ID)
print("MODEL_PATH =", MODEL_PATH)
print("cuda =", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu =", torch.cuda.get_device_name(0))
    print("bf16 supported =", torch.cuda.is_bf16_supported())


## 3. 准备一个很小的教学数据集

真实项目需要高质量数据、验证集和评估。这里故意用小数据，只演示链路。


In [ ]:
train_examples = [
    {"instruction": "解释 tokenizer 在大模型部署中的作用。", "answer": "tokenizer 负责把文本转成 token id，并把模型输出的 token id 解码回文本；在部署中还负责 chat template、pad/eos token 和 batch padding。"},
    {"instruction": "解释 prefill 和 decode 的区别。", "answer": "prefill 是一次性处理输入 prompt，建立初始 KV cache；decode 是逐 token 生成，每一步复用 KV cache 并追加新的 key/value。"},
    {"instruction": "解释 KV cache 为什么占显存。", "answer": "KV cache 保存每层 attention 的 key/value，大小随层数、batch、上下文长度、KV heads 和 head_dim 增长，在长上下文和高并发时会成为显存大头。"},
    {"instruction": "解释 LoRA 微调的核心思想。", "answer": "LoRA 冻结原模型权重，只在部分线性层旁边训练低秩矩阵，用很少的可训练参数改变模型行为，训练和保存成本都更低。"},
] * 8

len(train_examples), train_examples[0]


## 4. 加载 tokenizer 和基础模型


In [ ]:
from datasets import Dataset

try:
    from peft import LoraConfig, TaskType, get_peft_model
except ImportError as exc:
    raise ImportError(
        "PEFT import failed. Re-run the dependency installation cell above, "
        "then restart the notebook kernel and run the notebook again."
    ) from exc

from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments


tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=dtype if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False
cuda_memory("after base model load")


## 5. 加 LoRA adapter

Qwen/Llama 这类 decoder-only 模型常见 target modules 包括 attention projection 和 MLP projection。


In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## 6. 把 instruction/answer 转成模型训练文本

SFT 的关键是把样本渲染成模型的 chat template，然后做 causal language modeling。


In [ ]:
def format_example(example):
    messages = [
        {"role": "system", "content": "你是一个大模型部署面试辅导老师，回答要准确、简洁。"},
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["answer"]},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)


def tokenize_example(example):
    text = format_example(example)
    encoded = tokenizer(text, truncation=True, max_length=512)
    encoded["labels"] = encoded["input_ids"].copy()
    return encoded


dataset = Dataset.from_list(train_examples)
tokenized_dataset = dataset.map(tokenize_example, remove_columns=dataset.column_names)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

tokenized_dataset[0].keys(), len(tokenized_dataset[0]["input_ids"])


## 7. 开始一个短训练

这里的 `max_steps=20` 只用于演示流程，不代表真实收敛。


In [ ]:
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16

training_args = TrainingArguments(
    output_dir="outputs/qwen-lora-sft-demo",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    max_steps=20,
    learning_rate=2e-4,
    logging_steps=1,
    save_steps=20,
    report_to="none",
    bf16=use_bf16,
    fp16=use_fp16,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

trainer.train()


## 8. 保存和测试 adapter

LoRA 保存的是 adapter，不是完整模型权重。部署时通常需要 base model + adapter。


In [ ]:
adapter_dir = "outputs/qwen-lora-sft-demo/adapter"
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print("saved adapter to", adapter_dir)


In [ ]:
def generate_answer(question, max_new_tokens=120):
    messages = [
        {"role": "system", "content": "你是一个大模型部署面试辅导老师，回答要准确、简洁。"},
        {"role": "user", "content": question},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

print(generate_answer("LoRA 微调为什么省显存？"))


## 面试总结

- 全量微调会更新全部参数，显存和优化器状态成本高。
- LoRA 冻结 base model，只训练低秩 adapter，训练参数通常不到总参数的 1%。
- LoRA 适合风格、格式、领域知识的轻量适配；如果要注入大量新知识，数据质量和训练策略更关键。
